# IMDb Sentiment Analysis with Transformers

A zero-shot baseline: `distilbert-base-uncased-finetuned-sst-2-english` applied
to real held-out IMDb reviews from Hugging Face Datasets, with no fine-tuning.
Useful as a ceiling to compare the TF-IDF model in `sentiment_analysis.ipynb`
against.

### Before you start

Transformer dependencies are heavy and are deliberately kept out of the base
install. Set them up **once, from your shell** — not from inside the notebook:

```bash
pip install -r requirements-transformers.txt
```

The previous version of this notebook ran `%pip install -t ./.pydeps_transformers`
at runtime and then purged `sys.modules` and patched `sys.path` to pick the
vendored copy up. That made results irreproducible (the notebook was never run
top to bottom — its execution counts ran 27 to 34) and is confusing to read.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

In [ ]:
# Pull real IMDb reviews from Hugging Face Datasets
imdb = load_dataset("imdb")
df = pd.DataFrame(imdb["test"])
df = df.sample(2000, random_state=RANDOM_STATE).reset_index(drop=True)
df = df.rename(columns={"text": "review"})
df["sentiment"] = df["label"].map({1: "positive", 0: "negative"})

print(f"Loaded {len(df)} real held-out IMDb reviews from test split.")
df.head()

In [ ]:
# Load a pretrained sentiment model and tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, use_safetensors=True)
model.to(device)
model.eval()

print(f"Using device: {device}")

In [ ]:
# Run batched inference for speed and stability
def predict_batch(batch_texts):
    enc = tokenizer(
        batch_texts,
        truncation=True,
        max_length=512,
        padding=True,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        scores, labels = torch.max(probs, dim=-1)
    return labels.cpu().numpy().tolist(), scores.cpu().numpy().tolist()

texts = df["review"].tolist()
predicted_labels = []
predicted_scores = []

for i in tqdm(range(0, len(texts), 32), desc="Inferring"):
    batch = texts[i:i+32]
    labels, scores = predict_batch(batch)
    predicted_labels.extend(labels)
    predicted_scores.extend(scores)

df["pred_label"] = predicted_labels
df["pred_score"] = predicted_scores

df[["review", "sentiment", "label", "pred_label", "pred_score"]].head()

In [ ]:
# Evaluate performance
y_true = df["label"]
y_pred = df["pred_label"]

acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=["negative", "positive"]))

In [ ]:
# Confusion matrix plot
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"],
            yticklabels=["negative", "positive"])
plt.title("Transformer Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect predictions on real unseen reviews
sample_real = df.sample(5, random_state=RANDOM_STATE).copy()

for _, row in sample_real.iterrows():
    labels, scores = predict_batch([row["review"]])
    pred = "positive" if labels[0] == 1 else "negative"
    print(f"True: {row['sentiment']} | Pred: {pred} | Score: {scores[0]:.4f}")
    print(row["review"][:300].replace("\n", " ") + "...")
    print("-" * 100)